In [2]:

import json
from pathlib import Path
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms.functional as TF
import torch

In [3]:
torch.set_num_threads(96)
torch.set_num_interop_threads(96)

In [4]:
torch.get_num_threads()

96

In [5]:
from utils.midogpp import MIDOGpp

In [6]:
from utils.midesec import MiDeSec

In [7]:
import sqlite3
from pathlib import Path
from torch.utils.data import Dataset
from collections import defaultdict

class MiCCMCT(Dataset):
    """
    MITOS_WSI_CCMCT Dataset
    """

    def __init__(self, root_dir, sqlite_path, transforms=None):
        self.root_dir = Path(root_dir)
        self.transforms = transforms
        self.sqlite_path = sqlite_path
        self.source = "CCMCT"
        self.conn = sqlite3.connect(self.sqlite_path)
        self.cursor = self.conn.cursor()

        # -------------------------
        # 1. Load slides
        # -------------------------
        self.cursor.execute("SELECT uid, filename FROM Slides;")
        slides_data = self.cursor.fetchall()

        # slide_uid -> full image path
        self.slides = {
            uid: self.root_dir / filename
            for uid, filename in slides_data
        }

        # -------------------------
        # 2. Load class mapping
        # -------------------------
        self.cursor.execute("SELECT uid, name FROM Classes;")
        self.classes = dict(self.cursor.fetchall())

        # -------------------------
        # 3. Load annotations + labels
        # -------------------------
        self.cursor.execute("""
            SELECT A.uid, A.slide, L.class
            FROM Annotations A
            JOIN Annotations_label L
            ON A.uid = L.annoId
        """)
        ann_data = self.cursor.fetchall()
        # (anno_id, slide_uid, class_id)

        # -------------------------
        # 4. Load coordinates
        # -------------------------
        self.cursor.execute("""
            SELECT annoId, coordinateX, coordinateY
            FROM Annotations_coordinates
            ORDER BY annoId, orderIdx
        """)
        coord_data = self.cursor.fetchall()

        # Group coordinates by annotation
        coords_by_ann = defaultdict(list)
        for annoId, x, y in coord_data:
            coords_by_ann[annoId] = (x, y)
        
        self.samples = defaultdict(list)

        for anno_id, slide_uid, class_id in ann_data:
            if anno_id not in coords_by_ann:
                continue

            sample = {
                "label": class_id,
                "class_name": self.classes.get(class_id, None),
                "points": coords_by_ann[anno_id]
            }

            # self.samples[slide_uid].append(sample)
            if slide_uid in self.samples:
                self.samples[slide_uid]["instances"].append(sample)
            
            else:
                self.samples[slide_uid] = {
                    "image_id": slide_uid,
                    "slide_path": self.slides[slide_uid],
                    "instances" : [sample]
                }


        print(f"Loaded {len(ann_data)} annotations.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_data = self.samples[idx]
        return image_data


In [ ]:
import openslide
import random
import torch
import numpy as np
from torch.utils.data import Dataset
from typing import List

class PatchSampler(Dataset):
    def __init__(
        self,
        datasets: List[Dataset],
        patch_size: int = 512,
        bg_ratio: float = 0.3,
        epoch_length: int = 1000,
    ):
        self.datasets = datasets
        self.patch_size = patch_size
        self.bg_ratio = bg_ratio
        self.epoch_length = epoch_length
        self.turn = 0

    def __len__(self):
        return self.epoch_length

    def _next_dataset(self):
        self.turn = (self.turn+1)%len(self.datasets)
        return self.turn #pick the next dataset for every call

    @staticmethod
    def intersects(patch, bbox):
        px1, py1, px2, py2 = patch
        bx1, by1, bx2, by2 = bbox
        return not (px2 <= bx1 or px1 >= bx2 or py2 <= by1 or py1 >= by2)

    def _pick_patch_v1(self, dataset):
        # Sampling logic for MIDOG++ & MiDeSEC
        idx = random.randrange(len(dataset))
        sample = dataset[idx]

        image = sample["image"]      # [C, H, W]
        bboxes = sample["bboxes"]    # [(x1,y1,x2,y2)...]
        labels = sample["labels"]    # [1,2]
        image_id = sample["image_id"]

        _, H, W = image.shape
        P = self.patch_size

        # --- choose patch location ---
        if random.random() < self.bg_ratio:
            # background-biased random patch
            patch_x1 = random.randint(0, W - P)
            patch_y1 = random.randint(0, H - P)
        else:
            # bias around a random instance
            i = random.randrange(len(bboxes))
            x1, y1, x2, y2 = bboxes[i]
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            # sample patch so that (cx, cy) lies INSIDE, not necessarily at center
            min_x1: torch.Tensor = max(0, cx - P)
            max_x1: torch.Tensor = min(cx, W - P)

            min_y1: torch.Tensor = max(0, cy - P)
            max_y1: torch.Tensor = min(cy, H - P)

            # fallback safety (rare edge case when P > image dim or extreme clamp)
            if min_x1 > max_x1:
                patch_x1 = max(0, min(cx - P // 2, W - P))
            else:
                patch_x1 = random.randint(int(min_x1), int(max_x1))

            if min_y1 > max_y1:
                patch_y1 = max(0, min(cy - P // 2, H - P))
            else:
                patch_y1 = random.randint(int(min_y1), int(max_y1))

        patch_x2 = patch_x1 + P
        patch_y2 = patch_y1 + P
        patch_box = (patch_x1, patch_y1, patch_x2, patch_y2)

        patch = image[:, patch_y1:patch_y2, patch_x1:patch_x2]

        # --- collect all instances in patch ---
        patch_bboxes = []
        patch_labels = []

        for bb, lbl in zip(bboxes, labels):
            if self.intersects(patch_box, bb):
                x1, y1, x2, y2 = bb
                patch_bboxes.append([
                    x1 - patch_x1,
                    y1 - patch_y1,
                    x2 - patch_x1,
                    y2 - patch_y1,
                ])
                patch_labels.append(lbl)

        patch = patch.float().div_(255.0)
        patch = patch[:3]

        if len(patch_bboxes) == 0:
            return (patch,  {
                            "boxes": torch.zeros((0, 4), dtype=torch.float32),
                            "labels": torch.zeros((0,), dtype=torch.long),
                            "image_id": image_id,
                            })

        return (patch,  {
                        "boxes": torch.tensor(patch_bboxes, dtype=torch.float32),
                        "labels": torch.tensor(patch_labels, dtype=torch.long),
                        "image_id": image_id,
                        })

    def _pick_patch_v2(self, dataset):
        # ---- pick slide ----
        idx = random.randrange(len(dataset))
        sample = dataset[idx]

        image_id = sample["image_id"]
        slide = openslide.OpenSlide(sample["slide_path"])
        instances = sample["instances"]  # list of { "label": int, "points": (x, y) }

        W, H = slide.level_dimensions[0]
        P = self.patch_size

        # ---- choose patch location ----
        if random.random() < self.bg_ratio or len(instances) == 0:
            # background patch
            patch_x1 = random.randint(0, W - P)
            patch_y1 = random.randint(0, H - P)
        else:
            # center on random centroid
            inst = random.choice(instances)
            cx, cy = inst["points"]

           # sample patch so that (cx, cy) lies INSIDE, not necessarily at center
            min_x1: torch.Tensor = max(0, cx - P)
            max_x1: torch.Tensor = min(cx, W - P)
            min_y1: torch.Tensor = max(0, cy - P)
            max_y1: torch.Tensor = min(cy, H - P)

            # fallback safety (rare edge case when P > image dim or extreme clamp)
            if min_x1 > max_x1:
                patch_x1 = max(0, min(cx - P // 2, W - P))
            else:
                patch_x1 = random.randint(min_x1, max_x1)

            if min_y1 > max_y1:
                patch_y1 = max(0, min(cy - P // 2, H - P))
            else:
                patch_y1 = random.randint(min_y1, max_y1)

        patch_x2 = patch_x1 + P
        patch_y2 = patch_y1 + P

        # ---- read patch ----
        patch = slide.read_region(
            (patch_x1, patch_y1),
            0,
            (P, P)
        ).convert("RGB")

        patch = torch.from_numpy(np.array(patch)).permute(2, 0, 1).float().div_(255.0)

        # ---- collect centroids inside patch ----
        patch_points = []
        patch_labels = []

        for inst in instances:
            cx, cy = inst["points"]
            lbl = inst["label"]

            if patch_x1 <= cx < patch_x2 and patch_y1 <= cy < patch_y2:
                # shift to patch-local coordinates
                local_x = cx - patch_x1
                local_y = cy - patch_y1

                patch_points.append([local_x, local_y])
                patch_labels.append(lbl)

        slide.close()

        if len(patch_points) == 0:
            return (
                patch,
                {
                    "points": torch.zeros((0, 2), dtype=torch.float32),
                    "labels": torch.zeros((0,), dtype=torch.long),
                    "image_id": image_id,
                },
            )

        return (
            patch,
            {
                "points": torch.tensor(patch_points, dtype=torch.float32),
                "labels": torch.tensor(patch_labels, dtype=torch.long),
                "image_id": image_id,
            },
        )
        


    def __getitem__(self, _):
        turn_idx = self._next_dataset()
        dataset = self.datasets[turn_idx]
        if dataset.source == "CMC" or dataset.source == "CCMCT":
            return self._pick_patch_v2(dataset)
        else:
            return self._pick_patch_v1(dataset)
        

In [9]:
midogpp = MIDOGpp("/home/student4/PycharmProjects/DOG_MD/data/midogpp/", "/home/student4/PycharmProjects/DATA/MIDOGpp_IMAGES/MIDOGpp.json")

In [10]:
midesec = MiDeSec("/home/student4/PycharmProjects/DOG_MD/data/midesec")

In [11]:
miccmct = MiCCMCT("/home/student4/PycharmProjects/DATA/MITOS_WSI_CCMCT_WSI", "/home/student4/PycharmProjects/MITOS_WSI_CCMCT/databases/MITOS_WSI_CCMCT_MEL.sqlite")

Loaded 498379 annotations.


In [12]:
miccmct[1]

{'image_id': 1,
 'slide_path': PosixPath('/home/student4/PycharmProjects/DATA/MITOS_WSI_CCMCT_WSI/c91a842257ed2add5134.svs'),
 'instances': [{'label': 1,
   'class_name': 'granulocyte',
   'points': (21472, 54293)},
  {'label': 2, 'class_name': 'mitotic figure', 'points': (21472, 54293)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (22737, 51595)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (22737, 51595)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (23068, 55459)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (23068, 55459)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (23070, 55835)},
  {'label': 4, 'class_name': 'non-mitotic cell', 'points': (23070, 55835)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (23183, 55737)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (23183, 55737)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (23282, 55677)},
  {'label': 1, 'class_name': 'granulocyte', 'points': (23282, 5567

In [1]:
from utils.midogcmc import MiCMC

micmc = MiCMC("/home/student4/PycharmProjects/DATA/MITOS_WSI_CMC_WSI", "/home/student4/PycharmProjects/MITOS_WSI_CMC/databases/MITOS_WSI_CMC_MEL.sqlite")

Loaded 120778 annotations.


In [13]:
p1s = PatchSampler([midogpp, midesec, miccmct, micmc], patch_size=256)

In [15]:
p1s[1]

AttributeError: 'int' object has no attribute 'item'

In [ ]:
# class MemConsolidator(Dataset):
#     def __init__(self, samplers: List[PatchSampler], ratios: List[float]):
#         assert len(ratios) == len(samplers)
#         self.samplers = samplers
#         self.

In [ ]:
# TODO: Make a evaluation test sampler



In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2

# Load model and modify the head for custom classes
num_classes = 3

print(num_classes)

# Modify the box predictor to match the number of classes
model = fasterrcnn_resnet50_fpn_v2(weights=None,weights_backbone=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

3


In [ ]:
device = "cuda:0"

In [ ]:
import torch
def collate_fn(batch):
    images = []
    targets = []

    for img, target in batch:
        images.append(img)        # [C, H, W] → stackable
        target.pop('image_id')
        targets.append(target)    # dict with variable-length tensors

    images = torch.stack(images, dim=0)
    return images, targets

In [ ]:
train_loader = torch.utils.data.DataLoader(
    p1s, batch_size=8, shuffle=True, collate_fn=collate_fn
)

In [ ]:
import torch.optim as optim
from tqdm import tqdm
# Define optimizer
optimizer = optim.SGD(
    model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0005
)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Training loop
model.to(device)

num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    for images, targets in tqdm(train_loader):
        targets = [
                {k: v.to(device) for k, v in t.items()}
                for t in targets
            ]
        loss_dict = model(images.to(device), targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    lr_scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {losses.item()}")

100%|██████████| 125/125 [05:36<00:00,  2.69s/it]


Epoch 1, Loss: 0.12708432972431183


 17%|█▋        | 21/125 [00:57<04:44,  2.74s/it]


KeyboardInterrupt: 

In [ ]:
for images, targets in train_loader:
    image = images[0]
    ch = image[3]
    print(ch.min().item(), ch.max().item(), ch.mean().item())
    break

In [ ]:
import gc
torch.cuda.empty_cache()

In [ ]:
torch.save(model, 'fasterrcnn.pt')